# GRU: NER


In [ ]:
import collections
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from transformers import PreTrainedTokenizerFast, TrainingArguments

from core.args import DataArguments, ModelArguments
from core.datasets import (
    PAD_TOKEN,
    UNK_TOKEN,
    DataCollatorForTokenClassification,
    GeoNERDataset,
    bio_tags,
    build_word_tokenizer,
    load_uk_geo,
    row_spans,
)
from core.model import UniversalRNNTokenClassifier
from core.predict import infer
from helpers.config import CONFIG_DIR, PROCESSED_DIR, UK_GEO_PATH, parse_config

OUT_DIR = PROCESSED_DIR / "lecture" / "lstm_gru_ner"
OUT_DIR.mkdir(parents=True, exist_ok=True)

model_args, data_args, training_args = parse_config(
    CONFIG_DIR / "ner_gru.yaml",
    (ModelArguments, DataArguments, TrainingArguments),
)

## Data


In [ ]:
uk_geo = load_uk_geo(UK_GEO_PATH, data_args.train_rows, data_args.valid_rows)
train_df = uk_geo[uk_geo["split"] == "train"].reset_index(drop=True)
valid_df = uk_geo[uk_geo["split"] == "valid"].reset_index(drop=True)
example = train_df[train_df["loc_markers"].map(len) > 0].iloc[0]

print("Number of train sentences:", len(train_df))
print("Number of valid sentences:", len(valid_df))
print("Sentence example:", example["text"])

In [ ]:
words, tags = bio_tags(example["text"], row_spans(example))
pd.DataFrame({"word": words, "tag": tags})

In [ ]:
label_dist = collections.Counter(
    tag
    for text, spans in zip(train_df["text"], train_df.apply(row_spans, axis=1))
    for tag in bio_tags(text, spans)[1]
)

print("Unique labels:")
print(list(label_dist.keys()))

print("\nLabel distribution:")
for label, count in label_dist.items():
    print(f"{label}: {count}")

## Vocab


In [ ]:
tokenizer = build_word_tokenizer(
    train_df["text"].str.split(), specials=(UNK_TOKEN, PAD_TOKEN)
)

tokens = example["text"].split()
token_indices = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token indices:", token_indices)

In [ ]:
len(tokenizer)

In [ ]:
tokenizer.convert_tokens_to_ids(["поки", "що", "покищо", "покищо#", "<unk>", "<pad>"])

In [ ]:
token_freqs = collections.Counter(
    token for tokens in train_df["text"].str.split() for token in tokens
)

# Get the 30 most common tokens
top30 = token_freqs.most_common(30)
top30

## Dataset


In [ ]:
# Here we want to figure out empirically max_length

train_n_tokens = train_df["text"].str.split().str.len()

plt.title("Train N Tokens Distribution")
train_n_tokens.hist(bins=100)
plt.show()

max_length = np.quantile(train_n_tokens, data_args.max_length_quantile)
print(f"{data_args.max_length_quantile} Quantile of N tokens: {max_length}")

In [ ]:
ds_train = GeoNERDataset(train_df, tokenizer, max_length)
ds_valid = GeoNERDataset(valid_df, tokenizer, max_length)

In [ ]:
pad_token_id = tokenizer.pad_token_id
collator = DataCollatorForTokenClassification(pad_token_id, label_pad_token_id=-100)

dataloader = DataLoader(
    ds_train, batch_size=training_args.per_device_train_batch_size, collate_fn=collator
)

In [ ]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

## Inference

Trained by `scripts/train_ner.py`.


In [ ]:
out_dir = Path(training_args.output_dir)

if not (out_dir / "eval_results.json").exists():
    raise FileNotFoundError("Model was not trained yet.")

best_model = UniversalRNNTokenClassifier.from_pretrained(out_dir).eval()

tokenizer = PreTrainedTokenizerFast.from_pretrained(out_dir)

print("Eval:", json.loads((out_dir / "eval_results.json").read_text()))

In [ ]:
infer(best_model, "Привіт, як у тебе справи?", tokenizer)

In [ ]:
text = """Палата відділення щелепнолицевої хірургії по вулиці Зоологічній,
в палаті на кроватях лежать Василь Гнатович і Енгельс Гасанович"""
infer(best_model, text, tokenizer)

In [ ]:
tokenizer.convert_tokens_to_ids(
    ["Зоологічній", "Василь", "Гнатович", "Енгельс", "Гасанович"]
)

In [ ]:
text = """Недалеко от Богуслава, коло Росі, в довгому покрученому яру розкинулось село Семигори.
Яр в'ється гадюкою між крутими горами, між зеленими терасами;
од яру на всі боки розбіглись, неначе гілки дерева, глибокі рукави й поховались десь далеко в густих лісах."""
infer(best_model, text, tokenizer)